# Langgraph

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from typing import Annotated
from langchain_tavily import TavilySearch
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from typing_extensions import TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from langgraph.types import Command, interrupt

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]



### Tool condition may look like below: which decides on run time which tool next to go.
```python
def tools_condition(state):
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and len(last.tool_calls) > 0:
        return "tools"
    return END
```

In [22]:
graph_builder = StateGraph(State)
@tool
def human_assistance(query: str) -> str:
    """Request assitance from a human"""
    human_response = interrupt({"query": query})
    return human_response["data"]

tavily_search = TavilySearch(max_results=2)
tools = [tavily_search, human_assistance]
llm = ChatGroq(model="openai/gpt-oss-120b")
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    message = llm_with_tools.invoke(state["messages"])
    return {"messages": [message]}

graph_builder.add_node("chatbot", chatbot)
tool_node = ToolNode(tools=tools)

graph_builder.add_node("tools", tool_node)
graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition
)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")
graph = graph_builder.compile(checkpointer=MemorySaver())

In [23]:
config = {"configurable": {"thread_id": "1"}}
graph.invoke({"messages": "What is the AI news today?"}, config)

{'messages': [HumanMessage(content='What is the AI news today?', additional_kwargs={}, response_metadata={}, id='1e6013e6-96e7-451b-b0a5-9176f7aa3a6e'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What is the AI news today?" They likely want a summary of AI-related news for today (2026-10-29). Need up-to-date news. Use search tool. Query: AI news today 2026-10-29.', 'tool_calls': [{'id': 'fc_5883461e-7491-45e7-9650-68f9b0475fe2', 'function': {'arguments': '{"query":"AI news October 29 2026","search_depth":"advanced","time_range":"day"}', 'name': 'tavily_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 107, 'prompt_tokens': 344, 'total_tokens': 451, 'completion_time': 0.223138572, 'completion_tokens_details': {'reasoning_tokens': 55}, 'prompt_time': 0.015673998, 'prompt_tokens_details': None, 'queue_time': 0.394947578, 'total_time': 0.23881257}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_865